# 🛡️ options_hedge — do "outlier" stocks move more than their options priced in?

**Research only: no strategy, no backtest engine.** The idea: when a stock shows unusual activity
(news, unusual options trading, a volume spike), buy the stock with a put that protects against a
drop, sell the stock after a rise and keep the put in case the price comes back.

Stock + put pays like a call, and selling the stock after a rise leaves a put, so the package is a
**long-volatility** trade: it makes money only when the stock then moves MORE than its options
already priced in, plus the bid-ask spreads. Exit rules and a direction guess move profit around;
they cannot create it. This notebook runs the test written down in `README.md` BEFORE any screen
was scored:

1. **Data**: end-of-day option chains, news, daily bars, earnings dates.
2. **Data checks**: is the chain's price the stock's price, are the quotes in step, which names split.
3. **Screens**: the four "outlier" screens, and a look-ahead audit.
4. **Straddles**: the at-the-money straddle each name-day could have bought, and what is excluded.
5. **The test**: screened days against random days with the same dates and option prices.
6. **Pictures**.
7. **Verdict**: `CHECKS` table and one `SCORECARD` line.
8. **Holdout**: locked behind `READ_HOLDOUT`; read once, only for screens that pass discovery.

The first run builds caches in `var/options_hedge/` (chains ~15 min, SEC earnings ~5 min);
later runs take about a minute.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel (README "Quick start")
import importlib.util
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from trading_simple import plots, strategies

DIR = strategies.STRATEGIES_DIR / "options_hedge"
_spec = importlib.util.spec_from_file_location("options_hedge_research", DIR / "research.py")
rs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rs)

# --- the pre-registered windows (README) ---------------------------------------------------
DAILY_START = "2025-08-01"     # stock bars from here: 20 sessions of history before the chains
DISC = dict(months=("2025-10", "2026-04"), first_entry="2025-10-15", last_exp="2026-04-30")
HOLD = dict(months=("2026-05", "2026-08"), first_entry="2026-05-01", last_exp="2026-08-31")
READ_HOLDOUT = False           # True reads the locked window ONCE, for discovery-passing cells only
N_DRAWS = 20_000               # random sets per control (p resolves down to 1 / 20,001)
HOLD_ALPHA = 0.05              # holdout bar, divided by the number of cells taken there
pd.set_option("display.width", 160)

## 1 · Data

- **Chains** (`rs.chain_summaries`): every MarketData end-of-day chain file is reduced to one row
  per name and date: the stock price the chain was quoted against, the day's total option volume,
  and for both horizons the at-the-money straddle (expiry and strike rules in the README).
- **Daily bars** from the shared DB: adjusted close (returns), raw close and volume (split-adjusted
  share volume = volume × raw close ÷ adjusted close).
- **News**: MarketData headlines per name and session (the feed stamps a date, no time).
- **Earnings**: SEC 8-K Item 2.02 filings (the company's earnings release), stamped to the second;
  one accepted at or after 16:00 New York reacts on the next session. Foreign filers (6-K) have none
  and drop out.

In [ ]:
summ = rs.chain_summaries(*DISC["months"])
symbols = sorted(summ["symbol"].unique())
daily = rs.load_daily(symbols, DAILY_START, DISC["last_exp"])
sessions = rs.calendar(summ, daily)
news = rs.news_counts(symbols, sessions, *DISC["months"])
earnings = rs.fetch_earnings(symbols)
D = rs.assemble(summ, daily, news, earnings, sessions)
F = D["frames"]
print(f"chains: {len(summ):,} name-days, {len(symbols):,} names, "
      f"{summ['date'].min():%Y-%m-%d} → {summ['date'].max():%Y-%m-%d}")
print(f"daily bars: {daily['symbol'].nunique():,} names; calendar: {len(sessions)} sessions "
      f"{sessions[0]:%Y-%m-%d} → {sessions[-1]:%Y-%m-%d}")
print(f"news: {int(np.nansum(news.to_numpy())):,} headlines; "
      f"names with any: {(news.sum() > 0).sum():,}")
print(f"earnings: {earnings.dropna()['symbol'].nunique():,} names with SEC Item 2.02 dates")

## 2 · Data checks

Before any screen is scored:

- **Chain price vs DB**: the chain's stock price and the DB's adjusted close should move together
  day by day (they differ only on dividends and splits).
- **Breaks**: a jump of the ratio chain price ÷ adjusted close above 15% marks a split (or a bad
  bar); every straddle whose hold or trailing window touches one is excluded.
- **Put-call parity**: at the money, call − put ≈ price − strike. A large gap means the option quotes
  were not taken at the same moment as the stock price.
- **Outcomes**: the straddle settles on the chain's price on the expiry date; it must be there.

In [ ]:
r_chain = np.log(F["S"]).diff()
r_db = np.log(F["adj"]).diff()
gap = (r_chain - r_db).abs().where(~D["breaks"])
in_disc = ((sessions >= pd.Timestamp(DISC["first_entry"]))
           & (sessions <= pd.Timestamp(DISC["last_exp"])))
gap = gap[in_disc].stack()
brk = D["breaks"][in_disc].stack()
brk = brk[brk]
jumps = (np.log(F["S"] / F["adj"]).ffill().diff())[in_disc].stack().reindex(brk.index)
print(f"chain vs DB daily return gap: median {gap.median() * 1e4:.1f} bp, "
      f"> 1%: {(gap > 0.01).mean():.2%} of {len(gap):,} name-days")
print(f"breaks: {len(brk)} on {brk.index.get_level_values(1).nunique()} names")
print(pd.DataFrame({"ratio jump": np.exp(jumps).round(3)}).head(25).to_string())

rows30 = D["rows"]["30d"]
w30 = rows30[(rows30["entry"] >= pd.Timestamp(DISC["first_entry"]))
             & (rows30["exp"] <= pd.Timestamp(DISC["last_exp"]))]
call_mid, put_mid = (w30["cb"] + w30["ca"]) / 2, (w30["pb"] + w30["pa"]) / 2
parity = (call_mid - put_mid - (w30["S0"] - w30["K"])).abs() / w30["S0"]
found = w30["ST"].notna().mean()
print(f"put-call parity gap at the money (30d): median {parity.median():.2%}, "
      f"> 2%: {(parity > 0.02).mean():.2%}")
print(f"outcome price found at expiry (30d, discovery): {found:.2%}")

## 3 · The four screens

Each fires on day t from data through t's close, against the 20 sessions before t, and only on the
first day of an episode (not again within 5 sessions). `pool` is the set of name-days on which a
screen *could* have fired; the control draws from it.

The audit below recomputes every screen on data cut at a date and checks that nothing before the cut
changes: a screen that read a later session would differ.

In [ ]:
cut = sessions[sessions.searchsorted(pd.Timestamp("2026-02-02"))]
trunc = rs.screens(*(F[k].loc[:cut] for k in ("opt_vol", "vol_adj", "adj", "news")))
causal = all(D["screens"][k][i].loc[:cut].equals(trunc[k][i]) for k in rs.SCREENS for i in (0, 1))
print(f"look-ahead audit (cut {cut:%Y-%m-%d}): screens identical before the cut = {causal}")

win = (sessions >= pd.Timestamp("2025-10-14")) & (sessions <= pd.Timestamp(DISC["last_exp"]))
fires = {k: f[win] for k, (p, f) in D["screens"].items()}
pools = {k: p[win] for k, (p, f) in D["screens"].items()}
print(pd.DataFrame({k: {"pool name-days": int(pools[k].to_numpy().sum()),
                        "fires": int(fires[k].to_numpy().sum()),
                        "fire rate": fires[k].to_numpy().sum() / max(pools[k].to_numpy().sum(), 1)}
                    for k in rs.SCREENS}).T.to_string(float_format=lambda x: f"{x:.4f}"))
# how often two screens fire on the same name-day (share of the row screen's fires)
co = pd.DataFrame({a: {b: (fires[a] & fires[b]).to_numpy().sum() / max(fires[a].to_numpy().sum(), 1)
                       for b in rs.SCREENS} for a in rs.SCREENS}).T
print("\nshare of the row screen's fires that the column screen also fires on:")
print(co.round(2).to_string())

## 4 · Straddles: what each name-day could buy, and what is left out

A row = one name, one screen day t, the straddle bought at t+1's close. The funnel shows how many
rows each pre-registered exclusion removes. `elig` rows are scored; `earn_set` rows (the hold touches
an earnings release) are reported separately, for information only.

The table under it is the **base rate** of the whole eligible pool by how expensive the options were
(implied-move tercile within each name): `implied` = straddle mid ÷ price (the move the options
priced), `actual` = |price at expiry − price at entry| ÷ price.

In [ ]:
W = {h: rs.window_rows(D["rows"][h], D["react"], sessions, DISC["first_entry"], DISC["last_exp"])
     for h in rs.HORIZONS}
funnel = {}
for h, w in W.items():
    s1 = w["spread"] <= rs.MAX_SPREAD
    s2 = s1 & w["ST"].notna()
    s3 = s2 & ~w["broken"]
    funnel[h] = {"rows in window": len(w), "spread ≤ 10%": int(s1.sum()),
                 "outcome found": int(s2.sum()), "no split/break": int(s3.sum()),
                 "eligible (no earnings)": int(w["elig"].sum()),
                 "earnings set": int(w["earn_set"].sum()),
                 "names eligible": w.loc[w["elig"], "symbol"].nunique()}
print(pd.DataFrame(funnel).to_string())

base = []
for h, w in W.items():
    e = w[w["elig"]]
    g = e.assign(actual=(e["ST"] - e["S0"]).abs() / e["S0"]).groupby("tercile")
    t = g.agg(n=("r_ask", "size"), implied=("implied", "mean"), actual=("actual", "mean"),
              r_mid=("r_mid", "mean"), r_ask=("r_ask", "mean"), spread=("spread", "median"))
    t.loc["all"] = [len(e), e["implied"].mean(), ((e["ST"] - e["S0"]).abs() / e["S0"]).mean(),
                    e["r_mid"].mean(), e["r_ask"].mean(), e["spread"].median()]
    base.append(t.assign(horizon=h))
base = pd.concat(base)
print("\nbase rate of the eligible pool (no screen):")
print(base.round(3).to_string())

## 5 · The test (discovery window)

Per screen × horizon: `n` events, `r_ask` their mean straddle return bought at the ask, `ctrl_r_ask`
the mean of 20,000 random sets with the same number of picks per entry date and implied-move tercile,
`p` the share of random sets that did at least as well. `ratio` = mean payoff ÷ mid price (actual
move ÷ priced move, 1 = fair). `ci` = 95% interval of `r_ask` from resampling whole entry weeks.
`power_1.15` / `power_1.30`: how often fake screens planted with a 15% / 30% larger move beat the
control's 99.7% line.

**Pass** = `r_ask` > 0 AND `p` ≤ 0.003 AND still > 0 without the best week and without the top-5
names. A cell that fails the chance test with `power_1.15` < 0.8 is *inconclusive*, not a no.

`p_names` (added after the README was written, before scoring; a WARNING, never a gate): the control
treats events as independent, but one name can fire again a few sessions later with an overlapping
straddle. `p_names` bootstraps whole names over each event's excess above its cell's mean, so a
name's repeated events count as one piece of evidence.

In [ ]:
R = {h: rs.score_window(W[h], N_DRAWS) for h in rs.HORIZONS}


def table(results):
    rows = []
    for h, res in results.items():
        for k, r in res.items():
            verdict = "PASS" if r["passed"] else (
                "inconclusive" if not r["skill"] and r["power_1.15"] < 0.8 else "FAIL")
            cols = ("n", "names", "weeks", "r_ask", "ctrl_r_ask", "p", "p_names", "r_mid",
                    "ctrl_r_mid", "ratio", "no_best_week", "no_top5", "ci_lo", "ci_hi",
                    "power_1.15", "power_1.30")
            rows.append({"screen": k, "horizon": h, **{c: r[c] for c in cols}, "verdict": verdict})
    return pd.DataFrame(rows).set_index(["screen", "horizon"])


T = table(R)
print(T.round(3).to_string())

**Excess by month.** For each event: its return minus the mean of the pool rows in its own cell (same
entry date, same tercile), averaged per entry month. A real screen should be ahead in most months,
not in one.

In [ ]:
def monthly_excess(w, k):
    pool = w[w["elig"] & w[f"{k}_pool"]]
    cell_mean = pool.groupby(["entry", "tercile"])["r_ask"].transform("mean")
    ev = pool[pool[f"{k}_fire"]]
    x = (ev["r_ask"] - cell_mean[ev.index])
    return x.groupby(ev["entry"].dt.to_period("M")).agg(["mean", "size"])


M = pd.concat({(k, h): monthly_excess(W[h], k)["mean"] for h in rs.HORIZONS for k in rs.SCREENS},
              axis=1)
print(M.round(3).to_string())

**Earnings-window events (information only, not a gate).** The same test on name-days whose straddle
spans an earnings release: options are priced up for the release, so this is a different game.

In [ ]:
RE = {h: rs.score_window(W[h], N_DRAWS, earn=True) for h in rs.HORIZONS}
print(table(RE)[["n", "r_ask", "ctrl_r_ask", "p", "r_mid", "ratio", "no_best_week", "no_top5"]]
      .round(3).to_string())

## 6 · Pictures

**Left**: each cell's events (blue dot) against its control: the gray bar spans the random sets from
the 0.3% to the 99.7% quantile, the gray tick is their mean. A screen with skill puts its dot above
the bar. **Right**: the base rate of every eligible straddle by how expensive the options were.

In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4], horizontal_spacing=0.1,
                    subplot_titles=("screened days vs matched random days",
                                    "every eligible straddle"))
labels, obs, mean, lo, hi = [], [], [], [], []
for h, res in R.items():
    for k, r in res.items():
        labels.append(f"{k}<br>{h}")
        obs.append(r["r_ask"])
        mean.append(r["ctrl_r_ask"])
        lo.append(np.quantile(r["null"], rs.ALPHA))
        hi.append(np.quantile(r["null"], 1 - rs.ALPHA))
fig.add_trace(go.Scatter(x=labels, y=mean, mode="markers", name="random days (mean, 0.3–99.7%)",
                         marker={"color": plots._MUTED, "symbol": "line-ew-open", "size": 18,
                                 "line": {"color": plots._MUTED, "width": 2}},
                         error_y={"type": "data", "symmetric": False, "color": plots._MUTED,
                                  "array": np.subtract(hi, mean),
                                  "arrayminus": np.subtract(mean, lo)},
                         hovertemplate="%{x}<br>random mean %{y:+.1%}<extra></extra>"), 1, 1)
fig.add_trace(go.Scatter(x=labels, y=obs, mode="markers", name="screened days",
                         marker={"color": plots.SERIES[0], "size": 10},
                         hovertemplate="%{x}<br>screened %{y:+.1%}<extra></extra>"), 1, 1)
for h, col in zip(rs.HORIZONS, (plots.SERIES[1], plots.SERIES[2]), strict=True):
    b = base[(base["horizon"] == h) & (base.index != "all")]
    for j, m in enumerate(("r_mid", "r_ask")):
        fig.add_trace(go.Bar(x=[f"{h} {t}" for t in ("cheap", "middle", "dear")], y=b[m],
                             name=f"{h} at {m[2:]}",
                             marker={"color": col, "opacity": 1.0 if j else 0.45},
                             hovertemplate="%{x}<br>%{y:+.1%}<extra>" + f"{h} at {m[2:]}</extra>"),
                      1, 2)
fig.add_hline(y=0, line={"color": plots._INK, "width": 1}, row=1, col=1)
fig.add_hline(y=0, line={"color": plots._INK, "width": 1}, row=1, col=2)
fig.update_yaxes(tickformat="+.0%", title_text="mean straddle return (held to expiry)",
                 row=1, col=1)
fig.update_yaxes(tickformat="+.0%", row=1, col=2)
plots._base(fig, title="Do outlier days buy cheap movement? (discovery window)", height=480)
fig.update_layout(legend={"orientation": "h", "yanchor": "top", "y": -0.25, "x": 0,
                          "xanchor": "left"}, margin={"b": 120}, barmode="group")
fig.show()

## 7 · Verdict

In [ ]:
events = pd.concat([W[h][W[h]["elig"] & W[h][f"{k}_fire"]] for h in W for k in rs.SCREENS])
dated = earnings.dropna()["symbol"].nunique() / len(symbols)
after = bool((events["entry"] > events["t"]).all())
CHECKS = pd.DataFrame([
    ("chain price vs DB: median daily return gap < 10 bp", f"{gap.median() * 1e4:.1f} bp",
     gap.median() < 1e-3),
    ("put-call parity at the money: median gap < 0.5%", f"{parity.median():.2%}",
     parity.median() < 0.005),
    ("outcome price found at expiry ≥ 95%", f"{found:.1%}", found >= 0.95),
    ("names with SEC earnings dates ≥ 80%", f"{dated:.1%}", dated >= 0.8),
    ("screens unchanged by cutting later data", str(causal), causal),
    ("every straddle bought after its screen day", str(after), after),
    ("every cell has power ≥ 0.8 for a +30% move", f"min {T['power_1.30'].min():.2f}",
     T["power_1.30"].min() >= 0.8),
], columns=["check", "value", "ok"])
print(CHECKS.to_string(index=False))
passing = [(h, k) for (k, h) in T.index[T["verdict"] == "PASS"]]
print("\nverdict per cell:")
print(T["verdict"].unstack().to_string())
for h, k in passing:
    if not T.loc[(k, h), "p_names"] <= 0.05:
        print(f"WARNING {k}/{h}: name-clustered p = {T.loc[(k, h), 'p_names']:.3f} "
              "(the edge may be a few names' repeated events)")
print(f"\n{len(passing)} of {len(T)} cells pass discovery: {passing or 'none'}"
      + ("" if passing else " — the holdout stays unread."))
SCORECARD = {"notebook": "options_hedge", "window": "discovery",
             "checks_ok": bool(CHECKS["ok"].all()),
             "cells": {f"{k}/{h}": {"n": int(r["n"]), "r_ask": round(r["r_ask"], 4),
                                    "ctrl": round(r["ctrl_r_ask"], 4), "p": round(r["p"], 4),
                                    "p_names": round(r["p_names"], 4),
                                    "verdict": T.loc[(k, h), "verdict"]}
                       for h, res in R.items() for k, r in res.items()},
             "passing": [f"{k}/{h}" for h, k in passing]}
print("SCORECARD " + json.dumps(SCORECARD))

## 7b · Exploratory: more events, same question (NOT part of the verdict)

Every cell came out *inconclusive*: the 10% spread filter left 38–218 events per cell, too few to
see a +15% edge. Two looser looks at the same discovery data:

- **wider liquidity filters** (straddle spread ≤ 20% and ≤ 30% of mid): more names and days, and
  `r_ask` pays the wider spreads;
- **the liquid strike** (`selection = liquid`): the pre-registered rule takes the strike nearest the
  price with any two-sided quote, which is often a freshly listed $1 strike with no open interest; its
  wide quote then fails the 10% filter although the standard strike beside it is liquid (ABNB on
  2025-10-20: the 127 straddle quoted ~41% wide, the 125 one ~5%; found by an independent
  re-implementation of this test). This variant buys the nearest strike whose straddle passes the
  10% filter, as a trader would;
- **`any`**: the four screens pooled (a name-day where at least one fires), against random days
  drawn from name-days where at least one could have fired.

`excess` = events' `r_ask` minus the random sets' mean, with a ±1.96 sd band of the random sets. These
are extra looks: a narrow band around zero confirms there is no edge to find; a positive cell here
would only be a hypothesis for its own pre-registered holdout test, never a pass.

In [ ]:
EXPLORE = []
looks = [("nearest", ms, h, h) for ms in (0.10, 0.20, 0.30) for h in rs.HORIZONS]
looks += [("liquid", rs.MAX_SPREAD, h, v) for v, h in rs.LIQUID.items()]
for selection, max_spread, h, rows_key in looks:
    w = rs.window_rows(D["rows"][rows_key], D["react"], sessions, DISC["first_entry"],
                       DISC["last_exp"], max_spread=max_spread)
    w["any_pool"] = w[[f"{k}_pool" for k in rs.SCREENS]].any(axis=1)
    w["any_fire"] = w[[f"{k}_fire" for k in rs.SCREENS]].any(axis=1)
    for k, r in rs.score_window(w, N_DRAWS, cells=[*rs.SCREENS, "any"]).items():
        EXPLORE.append({"selection": selection, "max spread": max_spread, "horizon": h,
                        "screen": k, "pool": int(w["elig"].sum()),
                        **{c: r[c] for c in ("n", "r_ask", "ctrl_r_ask", "excess", "excess_lo",
                                             "excess_hi", "p", "p_names", "ratio",
                                             "power_1.15")}})
X = pd.DataFrame(EXPLORE).set_index(["selection", "max spread", "horizon", "screen"])
print(X.round(3).to_string())

fig = go.Figure()
for i, (sel, ms) in enumerate((("nearest", 0.10), ("nearest", 0.30), ("liquid", 0.10))):
    x = X.reset_index()
    x = x[(x["selection"] == sel) & (x["max spread"] == ms)]
    fig.add_trace(go.Scatter(
        x=x["excess"], y=x["screen"] + " · " + x["horizon"], mode="markers",
        name=f"{sel} strike, spread ≤ {ms:.0%}", marker={"color": plots.SERIES[i], "size": 9},
        error_x={"type": "data", "symmetric": False, "color": plots.SERIES[i],
                 "array": x["excess_hi"] - x["excess"], "arrayminus": x["excess"] - x["excess_lo"]},
        customdata=x[["n", "p"]],
        hovertemplate="%{y}<br>excess %{x:+.1%}<br>n %{customdata[0]}, p %{customdata[1]:.3f}"
                      "<extra></extra>"))
fig.add_vline(x=0, line={"color": plots._INK, "width": 1})
fig.update_xaxes(tickformat="+.0%",
                 title_text="screened minus random days: straddle return at the ask")
plots._base(fig, title="Excess over matched random days (95% band of the random sets)", height=520)
fig.update_layout(legend={"orientation": "h", "yanchor": "top", "y": -0.15, "x": 0,
                          "xanchor": "left"}, margin={"b": 100})
fig.show()

## 8 · Holdout (locked)

Straddles bought 2026-05-01 on and expiring by 2026-08-31, scored ONCE, only for cells that passed
discovery, each at one-sided p ≤ 0.05 ÷ (number of such cells) with mean `r_ask` > 0. The screens are
computed warm (trailing windows reach back into April). Leave `READ_HOLDOUT = False` until the
discovery result is final: reading the window spends it.

In [ ]:
if not passing:
    print("Nothing passed discovery: the holdout stays unread.")
elif not READ_HOLDOUT:
    print(f"Holdout locked (READ_HOLDOUT = False); it would score {passing}.")
else:
    summ_h = pd.concat([summ[summ["date"] >= "2026-03-01"], rs.chain_summaries(*HOLD["months"])],
                       ignore_index=True)
    daily_h = rs.load_daily(symbols, DAILY_START, HOLD["last_exp"])
    sessions_h = rs.calendar(summ_h, daily_h)
    news_h = rs.news_counts(symbols, sessions_h, "2026-03", HOLD["months"][1])
    D_h = rs.assemble(summ_h, daily_h, news_h, earnings, sessions_h)
    alpha = HOLD_ALPHA / len(passing)
    RH = {}
    for h, k in passing:
        w = rs.window_rows(D_h["rows"][h], D_h["react"], sessions_h,
                           HOLD["first_entry"], HOLD["last_exp"])
        RH.setdefault(h, {}).update(rs.score_window(w, N_DRAWS, cells=[k]))
    TH = table(RH)
    TH["holdout pass"] = (TH["r_ask"] > 0) & (TH["p"] <= alpha)
    print(f"holdout bar: r_ask > 0 and p ≤ {alpha:.4f}")
    print(TH.round(3).to_string())
    cells = {f"{k}/{h}": {"n": int(r["n"]), "r_ask": round(r["r_ask"], 4), "p": round(r["p"], 4),
                          "pass": bool(TH.loc[(k, h), "holdout pass"])}
             for h, res in RH.items() for k, r in res.items()}
    print("SCORECARD " + json.dumps({"notebook": "options_hedge", "window": "holdout",
                                     "cells": cells}))